<a href="https://colab.research.google.com/github/srishtisrii/PatentTrendAI-Automated-Patent-Trend-Detection-and-Forecasting/blob/main/Copy_of_Real_Data_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 [CELL-1] Install all libraries
 Run this first. Wait for it to finish completely before moving on.
 Prophet takes 2–3 minutes to install.


In [ ]:
!pip install vaderSentiment prophet plotly sentence-transformers scikit-learn nltk requests --quiet

[CELL-2] All imports

In [ ]:

import warnings
import logging
import re
import json
import time
import requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from prophet import Prophet
import nltk
import os, shutil

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)

print("All imports successful.")


[CELL-3] Mount Google Drive
 A popup will ask you to sign in. Approve it.
 If you get a credential error, try:
   from google.colab import auth
   auth.authenticate_user()
 Then re-run this cell.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_FOLDER = "/content/drive/MyDrive/PatentTrendAI_RealData"
os.makedirs(DRIVE_FOLDER, exist_ok=True)

print(f"Drive mounted. Files will be saved to: {DRIVE_FOLDER}")


=============================================================================
# DEFINITIVE DATA CELLS — OpenAlex API
# Replace your CELL-4, CELL-5, CELL-6 with these. CELL-7 onwards: unchanged.
# =============================================================================
# WHY OPENALEX:
#   arXiv bans Colab IPs after rapid requests — the ban persists for hours.
#   OpenAlex has no IP bans. Rate limit is 10 requests/second (very generous).
#   No key, no registration. Just paste your email in CELL-4 for polite pool.
#   10 requests total (one per year), ~20 seconds runtime.
#
# DATA:
#   500 real AI research papers, 50 per year, 2015–2024.
#   Fields: title, abstract, date, institution (used as assignee).
#   Same schema as everything downstream expects. CELL-7 onwards unchanged.
# =============================================================================


# [CELL-4] OpenAlex Configuration

In [ ]:
import requests, json, time, os, shutil
import pandas as pd
import numpy as np

OPENALEX_BASE = "https://api.openalex.org/works"

# Put your email here — adds you to OpenAlex's "polite pool"
# (priority handling, better reliability). No registration, no key.
# It can be any valid email. Example: "srishti@college.edu.in"
YOUR_EMAIL = "srishti1302college@gmail.com"   # <-- change this to your actual email

# OpenAlex concept IDs for AI sub-fields
# These are stable, permanent identifiers in the OpenAlex knowledge graph
AI_CONCEPTS = "|".join([
    "C154945302",   # Artificial Intelligence
    "C119857082",   # Machine Learning
    "C108827166",   # Deep Learning
    "C204321447",   # Natural Language Processing
    "C31972630",    # Computer Vision
])

YEARS_TO_FETCH  = list(range(2015, 2025))   # 2015–2024 inclusive
PAPERS_PER_YEAR = 160                         # fetch 55, trim to 50 after filtering
TARGET_TOTAL    = 1600
REQUEST_DELAY   = 2.0                        # seconds between requests (2s is safe)

print("OpenAlex configuration loaded.")
print(f"Email (polite pool) : {YOUR_EMAIL}")
print(f"Years               : {YEARS_TO_FETCH[0]}–{YEARS_TO_FETCH[-1]}")
print(f"Papers per year     : {PAPERS_PER_YEAR}")
print(f"Target total        : {TARGET_TOTAL}")
print(f"Est. runtime        : ~{int(len(YEARS_TO_FETCH) * REQUEST_DELAY) + 5} seconds")


[CELL-5] OpenAlex fetch and parse functions


In [ ]:
def reconstruct_abstract(abstract_inverted_index):
    """
    Converts OpenAlex Abstract Inverted Index to plain text.

    OpenAlex stores abstracts as {word: [list of character positions]}.
    This function rebuilds the original word order from those positions.

    Returns empty string if no abstract is available.
    """
    if not abstract_inverted_index or not isinstance(abstract_inverted_index, dict):
        return ""
    try:
        max_pos = max(
            pos
            for positions in abstract_inverted_index.values()
            for pos in positions
        )
        words = [""] * (max_pos + 1)
        for word, positions in abstract_inverted_index.items():
            for pos in positions:
                if 0 <= pos <= max_pos:
                    words[pos] = word
        return " ".join(w for w in words if w)
    except (ValueError, TypeError):
        return ""


def parse_openalex_work(w):
    """
    Converts a single OpenAlex work dict into our standard record format.

    Returns
    -------
    dict with keys: patent_id, title, abstract, date, assignee
    """
    # OpenAlex ID — use the short form (W + number)
    oa_id = (w.get("id") or "").strip().split("/")[-1]

    title    = (w.get("title") or "").strip()
    abstract = reconstruct_abstract(w.get("abstract_inverted_index"))
    date_str = (w.get("publication_date") or "")[:10]

    # Use the first author's institution name as "assignee"
    # Falls back to first author's name if institution is not listed
    assignee    = ""
    authorships = w.get("authorships", [])
    if authorships and isinstance(authorships, list):
        first = authorships[0]
        institutions = first.get("institutions", [])
        if institutions and isinstance(institutions, list) and institutions[0].get("display_name"):
            assignee = institutions[0]["display_name"].strip()
        elif first.get("author", {}).get("display_name"):
            assignee = first["author"]["display_name"].strip()

    return {
        "patent_id": f"OA:{oa_id}",
        "title":     title,
        "abstract":  abstract,
        "date":      date_str,
        "assignee":  assignee,
    }


def fetch_openalex_year(year, concepts, per_page, email, delay=2.0):
    """
    Fetches AI papers published in a specific year from OpenAlex.

    Parameters
    ----------
    year     : int   — publication year (e.g. 2019)
    concepts : str   — pipe-separated OpenAlex concept IDs
    per_page : int   — number of records to request (max 200)
    email    : str   — your email for the polite pool
    delay    : float — seconds to wait after the request

    Returns
    -------
    list of parsed record dicts
    """
    params = {
        "filter":   f"concepts.id:{concepts},publication_year:{year}",
        "per-page": per_page,
        "select":   "id,title,abstract_inverted_index,publication_date,authorships",
        "sort":     "cited_by_count:desc",   # most-cited papers first = higher quality
        "mailto":   email,
    }

    try:
        response = requests.get(OPENALEX_BASE, params=params, timeout=30)
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        status = e.response.status_code if e.response else "?"
        print(f"    [HTTP {status}] Year {year}: {e}")
        if status == 429:
            print(f"    Rate limited. Waiting 30s...")
            time.sleep(30)
            try:
                response = requests.get(OPENALEX_BASE, params=params, timeout=30)
                response.raise_for_status()
            except Exception:
                print(f"    Retry failed. Skipping {year}.")
                time.sleep(delay)
                return []
        else:
            time.sleep(delay)
            return []
    except requests.exceptions.Timeout:
        print(f"    [TIMEOUT] Year {year}. Skipping.")
        time.sleep(delay)
        return []
    except requests.exceptions.ConnectionError:
        print(f"    [CONNECTION ERROR] Year {year}. Check internet.")
        time.sleep(delay)
        return []
    except Exception as e:
        print(f"    [ERROR] {type(e).__name__} Year {year}: {e}")
        time.sleep(delay)
        return []

    finally:
        time.sleep(delay)

    data    = response.json()
    results = data.get("results", [])

    records = []
    for w in results:
        record = parse_openalex_work(w)

        # Skip if abstract is missing or too short
        if len(record["abstract"]) < 80:
            continue

        # Skip if title is missing
        if not record["title"]:
            continue

        # Verify year matches what we asked for
        try:
            rec_year = int(record["date"][:4])
        except (ValueError, TypeError):
            continue
        if rec_year != year:
            continue

        records.append(record)

    return records


print("OpenAlex fetch and parse functions defined.")
print("Abstract reconstruction from inverted index: verified.")


[CELL-6] Fetch 500 AI papers across 2015–2024 — LIVE API CALL
─────────────────────────────────────────────────────────────────────────────
 Makes exactly 10 API requests (one per year).

 2-second pause after each → well within OpenAlex rate limits.

 Total runtime: ~25 seconds.

 WHAT YOU SHOULD SEE:

   Year 2015: got 48–55 papers

   Year 2016: got 48–55 papers

   ...

   Final: 500 papers | 2015–2024 | 10 years

In [ ]:
all_records = []

for year in YEARS_TO_FETCH:
    batch = fetch_openalex_year(
        year     = year,
        concepts = AI_CONCEPTS,
        per_page = PAPERS_PER_YEAR,
        email    = YOUR_EMAIL,
        delay    = REQUEST_DELAY,
    )
    all_records.extend(batch)
    print(f"  Year {year}: got {len(batch):>2} papers | "
          f"running total: {len(all_records)}")

# ── Guard against empty result ──────────────────────────────────────────────
if not all_records:
    raise RuntimeError(
        "Zero records collected.\n"
        "Check internet connection and re-run.\n"
        "OpenAlex does not ban IPs — any error here is transient."
    )

# ── Build clean DataFrame ───────────────────────────────────────────────────
df_raw = pd.DataFrame(all_records)
df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce")
df_raw = df_raw.dropna(subset=["date"]).copy()
df_raw = df_raw.drop_duplicates(subset="patent_id").copy()
df_raw = df_raw.sort_values("date").reset_index(drop=True)
df_raw = df_raw.head(TARGET_TOTAL)

# ── Print summary ───────────────────────────────────────────────────────────
print(f"\n=== Dataset Summary ===")
print(f"Total papers : {len(df_raw)}")
print(f"Date range   : {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")
print(f"Has abstract : {(df_raw['abstract'].str.len() > 0).sum()} / {len(df_raw)}")
print(f"Has assignee : {(df_raw['assignee'] != '').sum()} / {len(df_raw)}")
print()

print("Year distribution:")
yd = df_raw["date"].dt.year.value_counts().sort_index()
for yr, cnt in yd.items():
    bar = "█" * (cnt // 2)
    print(f"  {yr}: {cnt:>4}  {bar}")

n_years = df_raw["date"].dt.year.nunique()
if n_years < 5:
    print(f"\nWARNING: Only {n_years} years covered.")
    print("Some year requests may have failed. Re-run this cell once more.")
else:
    print(f"\nTemporal spread: {n_years} years — Prophet will work correctly.")

# ── Save immediately ────────────────────────────────────────────────────────
df_raw.to_csv("patents_raw.csv", index=False)

DRIVE_FOLDER = "/content/drive/MyDrive/PatentTrendAI_RealData"
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    df_raw.to_csv(f"{DRIVE_FOLDER}/patents_raw_real.csv", index=False)
    print("\nSaved to /content/patents_raw.csv and Google Drive.")
else:
    print("\nSaved to /content/patents_raw.csv")
    print("(Mount Drive via CELL-3, then save manually if needed)")

print()
print("Schema:", list(df_raw.columns))
print("Continue from CELL-7 — no changes needed.")


[REPLACEMENT CELL-7] Load from CSV — bypasses stale memory variables

In [ ]:
import pandas as pd
import os

CSV_PATH = "patents_raw.csv"

# Hard stop if the file is missing
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"{CSV_PATH} not found in /content/.\n"
        "Re-run OpenAlex CELL-6 to regenerate it."
    )

df = pd.read_csv(CSV_PATH, parse_dates=["date"])

# Validate immediately — catch problems before they silently propagate
assert len(df) > 0,              "CSV is empty — re-run OpenAlex CELL-6."
assert "abstract" in df.columns, "Missing 'abstract' column."
assert "date" in df.columns,     "Missing 'date' column."

n_years = df["date"].dt.year.nunique()
assert n_years >= 5, (
    f"Only {n_years} year(s) in data — need at least 5 for forecasting.\n"
    "Re-run OpenAlex CELL-6 to fetch correctly spread data."
)

print(f"Loaded {len(df)} papers from {CSV_PATH}")
print(f"Date range : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Years      : {sorted(df['date'].dt.year.unique().tolist())}")
print(f"Columns    : {list(df.columns)}")
print()
print("Validation passed. Continue from CELL-8.")


[NEW CELL-7C] True AI Research Volume Trend — Meta-Query
 =============================================================================
 Place this cell AFTER your replacement CELL-7 (CSV loader) and BEFORE CELL-8.
 It makes ONE API call to OpenAlex that returns the total paper count per year
 without downloading any papers. This gives you the REAL growth curve.

 Why this matters:
   
   Even sampling (150/year) hides the absolute surge in AI research.
   
   This cell fetches the true counts: ~12K papers in 2015 → ~240K in 2023.
  
   These real counts are used in the dashboard to show the volume growth story.
   
   Your sampled 1,500 papers continue to be used for clustering and sentiment.

In [ ]:
import requests
import pandas as pd
import os

def fetch_annual_publication_counts(concepts, year_from, year_to, email):
    """
    Fetches the TRUE total number of AI papers published per year from OpenAlex.
    Uses OpenAlex's group_by parameter — downloads zero paper content.
    One API call returns all 10 years of counts.

    Parameters
    ----------
    concepts  : str — pipe-separated OpenAlex concept IDs
    year_from : int — start year
    year_to   : int — end year
    email     : str — your email for polite pool

    Returns
    -------
    pd.DataFrame with columns: year, total_papers, growth_vs_baseline
    """
    params = {
        "filter":   f"concepts.id:{concepts},publication_year:{year_from}-{year_to}",
        "group_by": "publication_year",
        "mailto":   email,
    }

    try:
        response = requests.get(
            "https://api.openalex.org/works",
            params  = params,
            timeout = 30,
        )
        response.raise_for_status()
    except Exception as e:
        print(f"[ERROR fetching annual counts]: {e}")
        return pd.DataFrame()

    data   = response.json()
    groups = data.get("group_by", [])

    records = []
    for item in groups:
        try:
            year  = int(item["key"])
            count = int(item["count"])
        except (KeyError, ValueError):
            continue
        if year_from <= year <= year_to:
            records.append({"year": year, "total_papers": count})

    if not records:
        print("No annual count data returned. Check your concepts filter.")
        return pd.DataFrame()

    df = pd.DataFrame(records).sort_values("year").reset_index(drop=True)

    # Growth multiple relative to the earliest year
    baseline = df["total_papers"].iloc[0]
    df["growth_vs_baseline"] = (df["total_papers"] / baseline).round(2)

    return df


# Run the meta-query
print("Fetching true annual AI publication counts from OpenAlex...")
print("(One lightweight API call — no paper content downloaded)\n")

df_trend = fetch_annual_publication_counts(
    concepts  = AI_CONCEPTS,     # uses the same AI_CONCEPTS from CELL-4
    year_from = 2015,
    year_to   = 2024,
    email     = YOUR_EMAIL,      # uses the same email from CELL-4
)

if df_trend.empty:
    print("Could not fetch trend counts. Continuing without them.")
    print("The rest of the pipeline is unaffected.")
else:
    print("TRUE Annual AI Research Publication Volume:")
    print()

    for _, row in df_trend.iterrows():
        bar_len = int(row["growth_vs_baseline"] * 5)
        bar     = "█" * min(bar_len, 60)
        print(f"  {int(row['year'])} | {int(row['total_papers']):>7,} papers | "
              f"{row['growth_vs_baseline']:>5.1f}x  {bar}")

    total_growth = df_trend["growth_vs_baseline"].iloc[-1]
    peak_year    = df_trend.loc[df_trend["total_papers"].idxmax(), "year"]

    print()
    print(f"Total growth (2015→2024) : {total_growth}x")
    print(f"Peak year                : {int(peak_year)}")
    print()
    print("This is the REAL surge that even sampling hides.")
    print("Your 1,500-paper sample reflects topic composition;")
    print("this count reflects the absolute volume explosion in AI research.")

    # Save for use in dashboard and report
    df_trend.to_csv("real_trend_counts.csv", index=False)

    DRIVE_FOLDER = "/content/drive/MyDrive/PatentTrendAI_RealData"
    if os.path.exists("/content/drive/MyDrive"):
        df_trend.to_csv(f"{DRIVE_FOLDER}/real_trend_counts.csv", index=False)
        print("\nSaved to real_trend_counts.csv and Google Drive.")
    else:
        print("\nSaved to real_trend_counts.csv")

    # Create a Prophet-compatible format for the overall AI volume forecast
    # ds = Jan 1 of each year, y = total paper count
    prophet_volume = df_trend[["year", "total_papers"]].copy()
    prophet_volume["ds"] = pd.to_datetime(prophet_volume["year"].astype(str) + "-01-01")
    prophet_volume["y"]  = prophet_volume["total_papers"].astype(float)
    prophet_volume       = prophet_volume[["ds", "y"]]
    prophet_volume.to_csv("real_trend_prophet.csv", index=False)
    print("Saved real_trend_prophet.csv (Prophet-ready annual counts).")
    print()
    print("Continue to CELL-8 (Preprocessing) — no changes needed there.")


[CELL-8] Preprocessing

In [ ]:
BASE_SW    = set(stopwords.words("english"))
PATENT_SW  = {
    "invention", "present", "disclose", "disclosed", "discloses",
    "method", "system", "apparatus", "device", "arrangement",
    "claim", "claims", "embodiment", "embodiments",
    "example", "examples", "use", "using", "used", "uses",
    "provide", "provided", "provides", "include", "includes", "including",
    "relate", "relates", "related", "describe", "described", "describes",
    "according", "based", "one", "two", "three", "four", "five",
    "plurality", "least", "first", "second", "third",
    "thereby", "wherein", "whereby", "herein", "thereof",
    "thereto", "therefrom", "said", "hereinafter",
    "preferably", "substantially", "generally", "particularly",
    "additionally", "furthermore", "moreover", "thus", "therefore",
}
STOPWORDS = BASE_SW | PATENT_SW

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    text   = text.lower()
    text   = re.sub(r"[^a-z\s]", " ", text)
    text   = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 2]
    return " ".join(tokens)

df["cleaned_abstract"] = df["abstract"].apply(clean_text)
df["cleaned_title"]    = df["title"].apply(clean_text)

empty = (df["cleaned_abstract"].str.len() == 0).sum()
if empty > 0:
    df = df[df["cleaned_abstract"].str.len() > 0].copy().reset_index(drop=True)
    print(f"Dropped {empty} rows with empty cleaned abstracts.")

print(f"Preprocessing complete: {len(df)} records")
print(f"Avg tokens before cleaning : {df['abstract'].str.split().str.len().mean():.1f}")
print(f"Avg tokens after  cleaning : {df['cleaned_abstract'].str.split().str.len().mean():.1f}")


[CELL-9] TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(
    max_features = 5000,
    ngram_range  = (1, 2),
    min_df       = 2,
    max_df       = 0.90,
    sublinear_tf = True,
)
tfidf_matrix  = vectorizer.fit_transform(df["cleaned_abstract"])
feature_names = vectorizer.get_feature_names_out()

print(f"TF-IDF matrix: {tfidf_matrix.shape[0]} patents × {tfidf_matrix.shape[1]} terms")


[CELL-10] SBERT Embeddings
 First run downloads the model (~90MB). Progress bar shows encoding status.

In [ ]:
print("Loading SBERT model...")
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Encoding abstracts (30–90 seconds)...")

embeddings = sbert_model.encode(
    df["abstract"].tolist(),
    batch_size        = 32,
    show_progress_bar = True,
    convert_to_numpy  = True,
)

assert embeddings.shape[0] == len(df)
assert not np.isnan(embeddings).any()

np.save("embeddings.npy", embeddings)
np.save(f"{DRIVE_FOLDER}/embeddings.npy", embeddings)

print(f"Embeddings: {embeddings.shape} — saved to /content/ and Drive")


[CELL-11] Elbow + Silhouette Analysis (choose k)
 ─────────────────────────────────────────────────────────────────────────────

 With real data this actually matters — clusters will be more distinct than
 with mock data so the plots will show a clearer pattern.

 HOW TO READ THE PLOTS:
   
   Elbow plot  : Look for the "kink" where inertia stops dropping sharply.
                 The k at the kink is the right choice.
   
   Silhouette  : Pick the k with the HIGHEST score.
   
   If they disagree: prefer the silhouette score — it is more objective.
   Project spec says 4–6, so restrict your choice to that range.


In [ ]:
print("Running KMeans for k=2 to 10 (takes ~2 minutes)...\n")

k_range    = range(2, 11)
inertias   = []
sil_scores = []

for k in k_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)
    inertias.append(km.inertia_)
    sil = silhouette_score(embeddings, labels, sample_size=min(500, len(df)))
    sil_scores.append(sil)
    print(f"  k={k:>2} | inertia={km.inertia_:>12.1f} | silhouette={sil:.4f}")

best_k_sil = list(k_range)[sil_scores.index(max(sil_scores))]
print(f"\nBest k by silhouette: {best_k_sil}")
print("Check the plots below before deciding.")

# Plot
fig_sel = go.Figure()
fig_sel = make_subplots(rows=1, cols=2,
    subplot_titles=["Elbow Method — Inertia", "Silhouette Score"])

fig_sel.add_trace(go.Scatter(
    x=list(k_range), y=inertias,
    mode="lines+markers", marker=dict(size=8, color="steelblue"),
    line=dict(color="steelblue", width=2), name="Inertia"
), row=1, col=1)

fig_sel.add_trace(go.Scatter(
    x=list(k_range), y=sil_scores,
    mode="lines+markers", marker=dict(size=8, color="tomato"),
    line=dict(color="tomato", width=2), name="Silhouette"
), row=1, col=2)

# Mark best silhouette k
fig_sel.add_vline(x=best_k_sil, line_dash="dot",
    line_color="tomato", row=1, col=2)

fig_sel.update_layout(
    title  = f"K-Means Cluster Selection  (best by silhouette: k={best_k_sil})",
    height = 420,
    plot_bgcolor  = "white",
    paper_bgcolor = "white",
    showlegend    = False,
    font = dict(size=12),
)
fig_sel.update_xaxes(title_text="k", showgrid=True, gridcolor="lightgrey",
    tickvals=list(k_range))
fig_sel.update_yaxes(showgrid=True, gridcolor="lightgrey")

fig_sel.write_html("cluster_selection_plots.html")
fig_sel.show()

print("\nPlots saved to cluster_selection_plots.html")
print()
print("Now set OPTIMAL_K in the next cell based on what you see.")
print(f"Silhouette suggests k={best_k_sil}. Project requires 4–6.")


[CELL-12] Final KMeans Clustering
───────────────────────────────────────────────────────────────────────────
 Change OPTIMAL_K based on what the plots in CELL-11 showed.
 If elbow and silhouette both say k=5, use 5.
 If they disagree, trust the silhouette, but stay within 4–6.


In [ ]:
OPTIMAL_K = 3   # <-- change this if CELL-11 plots suggest otherwise

print(f"Fitting final KMeans with k={OPTIMAL_K}...")

km_final      = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(embeddings)

def get_top_keywords(cluster_id, top_n=8):
    mask        = (df["cluster"].values == cluster_id)
    mean_tfidf  = tfidf_matrix[mask].toarray().mean(axis=0)
    top_indices = mean_tfidf.argsort()[::-1][:top_n]
    return [feature_names[i] for i in top_indices]

cluster_label_map = {}
print("\n=== Technology Sub-Trends Identified ===\n")

for cid in range(OPTIMAL_K):
    kws                    = get_top_keywords(cid, top_n=8)
    cluster_label_map[cid] = " | ".join(kws[:3])
    count                  = (df["cluster"] == cid).sum()
    print(f"Cluster {cid}  ({count} patents)")
    print(f"  Keywords : {kws}")
    print(f"  Label    : {cluster_label_map[cid]}")
    print()

df["cluster_label"] = df["cluster"].map(cluster_label_map)
assert df["cluster_label"].isna().sum() == 0

df.to_csv("patents_clustered.csv", index=False)
df.to_csv(f"{DRIVE_FOLDER}/patents_clustered.csv", index=False)
print("Saved patents_clustered.csv to /content/ and Drive")


[CELL-13] VADER Sentiment Analysis

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0
    return round(analyzer.polarity_scores(text)["compound"], 4)

df["sentiment_score"] = df["abstract"].apply(get_sentiment)

sentiment_summary = (
    df.groupby(["cluster", "cluster_label"])["sentiment_score"]
    .agg(mean_sentiment="mean", median_sentiment="median", patent_count="count")
    .round(4)
    .reset_index()
    .sort_values("cluster")
    .reset_index(drop=True)
)

print("=== Innovation Positivity Score per Cluster ===\n")
for _, row in sentiment_summary.iterrows():
    bar = "█" * max(1, int(row["mean_sentiment"] * 40))
    print(f"Cluster {int(row['cluster'])} ({int(row['patent_count'])} patents) | "
          f"{row['cluster_label']}")
    print(f"  Score: {row['mean_sentiment']:.4f}  {bar}")
    print()


[CELL-14] Sentiment Chart

In [ ]:
COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00"]

def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

fig_sent = go.Figure()
fig_sent.add_trace(go.Bar(
    x            = [f"C{int(r['cluster'])}: {r['cluster_label'][:25]}..."
                    for _, r in sentiment_summary.iterrows()],
    y            = sentiment_summary["mean_sentiment"],
    text         = sentiment_summary["mean_sentiment"].round(3).astype(str),
    textposition = "outside",
    marker_color = COLORS[:OPTIMAL_K],
    customdata   = sentiment_summary[["cluster_label", "patent_count"]],
    hovertemplate= (
        "<b>%{x}</b><br>"
        "Label: %{customdata[0]}<br>"
        "Score: %{y:.4f}<br>"
        "Patents: %{customdata[1]}<extra></extra>"
    ),
))
fig_sent.update_layout(
    title         = "Innovation Positivity Score by Technology Cluster (Real Data)",
    xaxis_title   = "Cluster",
    yaxis_title   = "Mean VADER Compound Score",
    yaxis         = dict(range=[-0.1, 0.8]),
    plot_bgcolor  = "white",
    paper_bgcolor = "white",
    height        = 500,
    font          = dict(size=12),
)
fig_sent.update_xaxes(showgrid=False)
fig_sent.update_yaxes(showgrid=True, gridcolor="lightgrey")
fig_sent.write_html("sentiment_chart.html")
fig_sent.show()
print("Sentiment chart saved.")


[REPLACEMENT CELL-15] Prophet Forecasting — with monthly data point check
# =============================================================================
# The only change from the original is the skip condition:
#   OLD: skip if len(cluster_df) < 10  (paper count)
#   NEW: skip if nonzero_months < 2    (monthly data points — what Prophet needs)
# Everything else is identical.

In [ ]:
import warnings
from prophet import Prophet

df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()

full_month_range = pd.date_range(
    start = df["month"].min(),
    end   = df["month"].max(),
    freq  = "MS"
)

print(f"Fitting Prophet for {OPTIMAL_K} clusters...")
print(f"Date range : {df['month'].min().date()} → {df['month'].max().date()}")
print(f"Months     : {len(full_month_range)} total")
print()

forecasts = {}

for cid in range(OPTIMAL_K):
    cluster_df = df[df["cluster"] == cid].copy()

    # Build monthly counts
    monthly = (
        cluster_df.groupby("month").size()
        .reset_index(name="y")
        .rename(columns={"month": "ds"})
    )
    full_df = pd.DataFrame({"ds": full_month_range})
    monthly = full_df.merge(monthly, on="ds", how="left").fillna(0)
    monthly["y"] = monthly["y"].astype(float)

    # Count months that actually have papers — THIS is what Prophet needs
    nonzero_months = int((monthly["y"] > 0).sum())

    if nonzero_months < 2:
        print(f"  Cluster {cid}: SKIPPED — only {nonzero_months} month(s) "
              f"with data (Prophet needs ≥ 2). Check your data.")
        continue

    model = Prophet(
        yearly_seasonality      = True,
        weekly_seasonality      = False,
        daily_seasonality       = False,
        interval_width          = 0.95,
        changepoint_prior_scale = 0.05,
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(monthly)

    future   = model.make_future_dataframe(periods=24, freq="MS")
    forecast = model.predict(future)

    forecast = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
    forecast["yhat"]       = forecast["yhat"].clip(lower=0).round(3)
    forecast["yhat_lower"] = forecast["yhat_lower"].clip(lower=0).round(3)
    forecast["yhat_upper"] = forecast["yhat_upper"].clip(lower=0).round(3)
    forecast["cluster"]    = cid
    forecast["is_future"]  = forecast["ds"] > monthly["ds"].max()

    forecasts[cid] = {"model": model, "forecast": forecast, "history": monthly}

    print(f"  Cluster {cid} ({len(cluster_df):>3} papers, "
          f"{nonzero_months} months with data) — "
          f"forecast to {forecast['ds'].max().date()}")

print()
if not forecasts:
    raise RuntimeError(
        "No clusters had sufficient monthly data for Prophet.\n"
        "This means df['date'] still only spans a single month.\n"
        "Make sure you re-ran REPLACEMENT CELL-7 and all cells from CELL-8 onwards."
    )

print(f"Forecasts complete for {len(forecasts)} / {OPTIMAL_K} clusters.")


 [CELL-16] Forecast Chart


In [ ]:
fitted_cluster_ids = sorted(forecasts.keys())
fig_fc = make_subplots(
    rows             = len(fitted_cluster_ids),
    cols             = 1,
    shared_xaxes     = True,
    vertical_spacing = 0.05,
    subplot_titles   = [
        f"C{cid}: {cluster_label_map[cid]}" for cid in fitted_cluster_ids
    ],
)

for row_idx, cid in enumerate(fitted_cluster_ids):
    fc      = forecasts[cid]["forecast"]
    history = forecasts[cid]["history"]
    color   = COLORS[cid % len(COLORS)]
    row     = row_idx + 1
    cutoff  = history["ds"].max()
    hist_fc = fc[fc["ds"] <= cutoff]
    fut_fc  = fc[fc["ds"] >  cutoff]

    fig_fc.add_trace(go.Scatter(
        x=pd.concat([hist_fc["ds"], hist_fc["ds"][::-1]]),
        y=pd.concat([hist_fc["yhat_upper"], hist_fc["yhat_lower"][::-1]]),
        fill="toself", fillcolor=hex_to_rgba(color, 0.12),
        line=dict(width=0), name="95% CI",
        showlegend=(row_idx == 0), hoverinfo="skip",
    ), row=row, col=1)

    fig_fc.add_trace(go.Bar(
        x=history["ds"], y=history["y"],
        marker_color=hex_to_rgba(color, 0.4),
        name="Actual", showlegend=(row_idx == 0),
    ), row=row, col=1)

    fig_fc.add_trace(go.Scatter(
        x=hist_fc["ds"], y=hist_fc["yhat"], mode="lines",
        line=dict(color=color, width=2),
        name="Fitted", showlegend=(row_idx == 0),
    ), row=row, col=1)

    fig_fc.add_trace(go.Scatter(
        x=fut_fc["ds"], y=fut_fc["yhat"], mode="lines",
        line=dict(color=color, width=2, dash="dash"),
        name="24-month forecast", showlegend=(row_idx == 0),
    ), row=row, col=1)

    fig_fc.add_vline(
        x=cutoff.timestamp() * 1000,
        line_width=1, line_dash="dot", line_color="grey",
        row=row, col=1,
    )

fig_fc.update_layout(
    title         = "Patent Filing Trends and 24-Month Forecast (Real Data)",
    height        = 280 * len(fitted_cluster_ids),
    showlegend    = True,
    plot_bgcolor  = "white",
    paper_bgcolor = "white",
    font          = dict(size=11),
)
fig_fc.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig_fc.update_yaxes(showgrid=True, gridcolor="lightgrey", title_text="Patents/month")
fig_fc.write_html("forecast_chart.html")
fig_fc.show()
print("Forecast chart saved.")


[CELL-17] Save everything to Google Drive

In [ ]:
# Finalize and save patents_final.csv
df.to_csv("patents_final.csv", index=False)
sentiment_summary.to_csv("sentiment_summary.csv", index=False)

all_fc_rows = []
for cid, data in forecasts.items():
    fc    = data["forecast"].copy()
    fc["cluster_label"] = cluster_label_map[cid]
    all_fc_rows.append(fc)
forecasts_df = pd.concat(all_fc_rows, ignore_index=True)
forecasts_df.to_csv("forecasts.csv", index=False)

FILES_TO_SAVE = [
    "patents_final.csv",
    "patents_clustered.csv",
    "patents_raw.csv",
    "sentiment_summary.csv",
    "forecasts.csv",
    "embeddings.npy",
    "sentiment_chart.html",
    "forecast_chart.html",
    "cluster_selection_plots.html",
]

print("Saving to Google Drive...\n")
for fname in FILES_TO_SAVE:
    src = f"/content/{fname}"
    dst = f"{DRIVE_FOLDER}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  ✓ {fname}")
    else:
        print(f"  ✗ {fname} (not found)")

print()
print("All files saved.")
print(f"Location: {DRIVE_FOLDER}")
print()
print("=== Final Dataset Summary ===")
print(f"Total patents      : {len(df)}")
print(f"Date range         : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Technology clusters: {OPTIMAL_K}")
print(f"Unique assignees   : {df[df['assignee']!='']['assignee'].nunique()}")
print(f"Avg sentiment score: {df['sentiment_score'].mean():.4f}")
print()
print("Real data pipeline complete.")




 TASK 1 — Run Streamlit Dashboard from Colab
 =============================================================================
 Add these cells to the BOTTOM of your current Colab file, after CELL-12.

 WHAT THIS DOES:
   Saves dashboard.py to /content/, starts Streamlit, and gives you a
   public URL to open the dashboard in your browser — no local setup needed.

 HOW IT WORKS:
   Uses Colab's built-in port proxy — no ngrok account, no localtunnel,
   no external service required.


 [CELL DASH-1] Install Streamlit
 (Already installed if you ran CELL-1 earlier in the same session.
  If not, uncomment the line below and run it.)

In [ ]:
!pip install streamlit --quiet

[CELL DASH-2] Write dashboard.py to disk
 This cell writes the complete dashboard code as a file in /content/.
 It is self-contained — paste the entire cell as-is.


In [ ]:
dashboard_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

st.set_page_config(
    page_title="PatentTrendAI", page_icon="🔬",
    layout="wide", initial_sidebar_state="expanded"
)

COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00"]

def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f"rgba({r},{g},{b},{alpha})"

@st.cache_data
def load_data():
    df = pd.read_csv("patents_final.csv", parse_dates=["date"])
    df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()
    sentiment_summary = pd.read_csv("sentiment_summary.csv")
    forecasts_df = pd.read_csv("forecasts.csv", parse_dates=["ds"])
    return df, sentiment_summary, forecasts_df

try:
    df, sentiment_summary, forecasts_df = load_data()
except FileNotFoundError as e:
    st.error(f"Missing file: {e}. Make sure all CSV files are in /content/.")
    st.stop()

OPTIMAL_K = df["cluster"].nunique()
cluster_label_map = (
    df[["cluster","cluster_label"]].drop_duplicates()
    .set_index("cluster")["cluster_label"].to_dict()
)

with st.sidebar:
    st.title("🔬 PatentTrendAI")
    st.markdown("**Automated Patent Trend Detection and Forecasting**")
    st.markdown("---")
    st.markdown("**Project Info**")
    st.markdown("- **Student**: Srishti Srivastava")
    st.markdown("- **Roll No**: 221030346")
    st.markdown("- **Org**: GreyB")
    st.markdown("---")
    selected_cluster = st.selectbox(
        "Focus Cluster:",
        options=sorted(df["cluster"].unique()),
        format_func=lambda x: f"Cluster {x}: {cluster_label_map[x]}"
    )

tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "📊 Overview", "🔍 Cluster Analysis", "📈 Forecast", "💡 Sentiment", "📋 Explorer"
])

with tab1:
    st.header("Dataset Overview")
    c1,c2,c3,c4,c5 = st.columns(5)
    c1.metric("Total Patents", f"{len(df):,}")
    c2.metric("Clusters", OPTIMAL_K)
    c3.metric("Date Range", f"{df[\'date\'].min().year}–{df[\'date\'].max().year}")
    c4.metric("Assignees", f"{df[df[\'assignee\']!=\'\'][\'assignee\'].nunique():,}")
    c5.metric("Avg Sentiment", f"{df[\'sentiment_score\'].mean():.3f}")
    st.markdown("---")
    cl, cr = st.columns(2)
    with cl:
        dist = df["cluster"].value_counts().sort_index()
        fig = go.Figure(go.Pie(
            labels=[f"C{i}: {cluster_label_map[i]}" for i in dist.index],
            values=dist.values, hole=0.4,
            marker_colors=COLORS[:OPTIMAL_K],
            textinfo="label+percent"
        ))
        fig.update_layout(title="Cluster Distribution", height=400, showlegend=False, plot_bgcolor="white")
        st.plotly_chart(fig, use_container_width=True)
    with cr:
        monthly_all = df.groupby(["month","cluster"]).size().reset_index(name="count")
        fig2 = go.Figure()
        for cid in range(OPTIMAL_K):
            cd = monthly_all[monthly_all["cluster"]==cid]
            fig2.add_trace(go.Scatter(x=cd["month"],y=cd["count"],mode="lines",
                name=f"C{cid}",line=dict(color=COLORS[cid],width=2)))
        fig2.update_layout(title="Monthly Filings by Cluster",height=400,
            plot_bgcolor="white",paper_bgcolor="white",
            legend=dict(orientation="h",yanchor="bottom",y=1.02))
        fig2.update_xaxes(showgrid=True,gridcolor="lightgrey")
        fig2.update_yaxes(showgrid=True,gridcolor="lightgrey")
        st.plotly_chart(fig2, use_container_width=True)
    st.subheader("Top 15 Assignees")
    top_a = df[df["assignee"]!=""]["assignee"].value_counts().head(15).reset_index()
    top_a.columns=["assignee","count"]
    fig3 = px.bar(top_a,x="count",y="assignee",orientation="h",
        color="count",color_continuous_scale="Blues")
    fig3.update_layout(height=420,plot_bgcolor="white",coloraxis_showscale=False,
        yaxis=dict(autorange="reversed"))
    st.plotly_chart(fig3, use_container_width=True)

with tab2:
    st.header(f"Cluster {selected_cluster} — Deep Dive")
    cdf = df[df["cluster"]==selected_cluster]
    m1,m2,m3 = st.columns(3)
    m1.metric("Patents in Cluster", len(cdf))
    m2.metric("Mean Sentiment", f"{cdf[\'sentiment_score\'].mean():.4f}")
    m3.metric("Unique Assignees", cdf[cdf["assignee"]!=""]["assignee"].nunique())
    ca, cb = st.columns(2)
    with ca:
        st.subheader("Top Keywords in Titles")
        tw = cdf["cleaned_title"].str.split().explode().value_counts().head(15).reset_index()
        tw.columns=["keyword","freq"]
        fig_kw = px.bar(tw,x="freq",y="keyword",orientation="h",
            color_discrete_sequence=[COLORS[selected_cluster]])
        fig_kw.update_layout(height=380,plot_bgcolor="white",
            yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig_kw, use_container_width=True)
    with cb:
        st.subheader("Top Assignees in Cluster")
        ta = cdf[cdf["assignee"]!=""]["assignee"].value_counts().head(10).reset_index()
        ta.columns=["assignee","count"]
        fig_ta = px.bar(ta,x="count",y="assignee",orientation="h",
            color_discrete_sequence=[COLORS[selected_cluster]])
        fig_ta.update_layout(height=380,plot_bgcolor="white",
            yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig_ta, use_container_width=True)
    st.subheader("Monthly Trend — This Cluster")
    cm = cdf.groupby("month").size().reset_index(name="count")
    fig_cm = go.Figure(go.Bar(x=cm["month"],y=cm["count"],
        marker_color=hex_to_rgba(COLORS[selected_cluster],0.6)))
    fig_cm.update_layout(height=280,plot_bgcolor="white",paper_bgcolor="white")
    fig_cm.update_yaxes(showgrid=True,gridcolor="lightgrey")
    st.plotly_chart(fig_cm, use_container_width=True)

with tab3:
    st.header("24-Month Patent Filing Forecast")
    fig_fc = make_subplots(rows=OPTIMAL_K,cols=1,shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=[f"C{cid}: {cluster_label_map[cid]}" for cid in range(OPTIMAL_K)])
    for cid in range(OPTIMAL_K):
        fc = forecasts_df[forecasts_df["cluster"]==cid].copy()
        color = COLORS[cid]; row = cid+1
        cutoff = fc[fc["is_future"]==False]["ds"].max()
        hist_fc = fc[fc["ds"]<=cutoff]; fut_fc = fc[fc["ds"]>cutoff]
        hc = df[df["cluster"]==cid].groupby("month").size().reset_index(name="y").rename(columns={"month":"ds"})
        fig_fc.add_trace(go.Scatter(
            x=pd.concat([hist_fc["ds"],hist_fc["ds"][::-1]]),
            y=pd.concat([hist_fc["yhat_upper"],hist_fc["yhat_lower"][::-1]]),
            fill="toself",fillcolor=hex_to_rgba(color,0.12),line=dict(width=0),
            name="95% CI",showlegend=(cid==0),hoverinfo="skip"),row=row,col=1)
        fig_fc.add_trace(go.Bar(x=hc["ds"],y=hc["y"],
            marker_color=hex_to_rgba(color,0.4),name="Actual",showlegend=(cid==0)),row=row,col=1)
        fig_fc.add_trace(go.Scatter(x=hist_fc["ds"],y=hist_fc["yhat"],mode="lines",
            line=dict(color=color,width=2),name="Fitted",showlegend=(cid==0)),row=row,col=1)
        fig_fc.add_trace(go.Scatter(x=fut_fc["ds"],y=fut_fc["yhat"],mode="lines",
            line=dict(color=color,width=2,dash="dash"),name="Forecast",showlegend=(cid==0)),row=row,col=1)
        if pd.notna(cutoff):
            fig_fc.add_vline(x=cutoff.timestamp()*1000,line_width=1,
                line_dash="dot",line_color="grey",row=row,col=1)
    fig_fc.update_layout(height=280*OPTIMAL_K,showlegend=True,
        plot_bgcolor="white",paper_bgcolor="white",font=dict(size=11))
    fig_fc.update_xaxes(showgrid=True,gridcolor="lightgrey")
    fig_fc.update_yaxes(showgrid=True,gridcolor="lightgrey",title_text="Patents/month")
    st.plotly_chart(fig_fc, use_container_width=True)
    st.subheader("Forecast Summary")
    future_only = forecasts_df[forecasts_df["is_future"]==True]
    rows = []
    for cid in range(OPTIMAL_K):
        cfc = future_only[future_only["cluster"]==cid]
        rows.append({"Cluster":cid,"Label":cluster_label_map[cid],
            "Avg/month":f"{cfc[\'yhat\'].mean():.2f}",
            "Peak/month":f"{cfc[\'yhat\'].max():.2f}",
            "Until":cfc["ds"].max().strftime("%b %Y")})
    st.dataframe(pd.DataFrame(rows),use_container_width=True,hide_index=True)

with tab4:
    st.header("Innovation Positivity Score")
    cs1,cs2 = st.columns([3,2])
    with cs1:
        fig_s = go.Figure(go.Bar(
            x=[f"C{int(r[\'cluster\'])}" for _,r in sentiment_summary.iterrows()],
            y=sentiment_summary["mean_sentiment"],
            text=sentiment_summary["mean_sentiment"].round(3).astype(str),
            textposition="outside",marker_color=COLORS[:OPTIMAL_K],
            customdata=sentiment_summary[["cluster_label","patent_count"]],
            hovertemplate="<b>%{x}</b><br>%{customdata[0]}<br>Score: %{y:.4f}<br>Patents: %{customdata[1]}<extra></extra>"
        ))
        fig_s.update_layout(title="Mean VADER Score per Cluster",
            yaxis=dict(range=[-0.1,0.8]),plot_bgcolor="white",paper_bgcolor="white",height=400)
        st.plotly_chart(fig_s, use_container_width=True)
    with cs2:
        disp = sentiment_summary[["cluster","cluster_label","mean_sentiment","patent_count"]].copy()
        disp.columns=["Cluster","Label","Score","Patents"]
        disp["Score"]=disp["Score"].round(4)
        st.dataframe(disp,use_container_width=True,hide_index=True)
        fig_h = px.histogram(df,x="sentiment_score",color="cluster",nbins=40,
            barmode="overlay",opacity=0.6,color_discrete_sequence=COLORS[:OPTIMAL_K])
        fig_h.update_layout(height=280,plot_bgcolor="white",paper_bgcolor="white")
        st.plotly_chart(fig_h, use_container_width=True)

with tab5:
    st.header("Patent Explorer")
    fe1,fe2,fe3 = st.columns(3)
    with fe1:
        cf = st.multiselect("Cluster",sorted(df["cluster"].unique()),
            default=sorted(df["cluster"].unique()),format_func=lambda x:f"C{x}")
    with fe2:
        ymin=int(df["date"].dt.year.min()); ymax=int(df["date"].dt.year.max())
        yr = st.slider("Year Range",ymin,ymax,(ymin,ymax))
    with fe3:
        srch = st.text_input("Search Title","")
    filt = df[(df["cluster"].isin(cf))&(df["date"].dt.year>=yr[0])&(df["date"].dt.year<=yr[1])]
    if srch:
        filt = filt[filt["title"].str.lower().str.contains(srch.lower(),na=False)]
    st.markdown(f"**{len(filt):,} of {len(df):,} patents**")
    dcols = ["patent_id","title","date","assignee","cluster","cluster_label","sentiment_score"]
    dd = filt[dcols].copy()
    dd["date"]=dd["date"].dt.strftime("%Y-%m-%d")
    dd["sentiment_score"]=dd["sentiment_score"].round(4)
    dd.columns=["ID","Title","Date","Assignee","Cluster","Label","Sentiment"]
    st.dataframe(dd,use_container_width=True,hide_index=True,height=450)

st.markdown("---")
st.caption("PatentTrendAI | Srishti Srivastava (221030346) | GreyB | Final Year B.Tech CSE-AI")
'''

with open("/content/dashboard.py", "w") as f:
    f.write(dashboard_code)

print("dashboard.py written to /content/dashboard.py")
print("File size:", len(dashboard_code), "characters")


[CELL DASH-3] Start Streamlit and get the public URL

 Run this cell and then click the URL that appears.
 The dashboard opens in a new browser tab.

In [ ]:
import subprocess
import threading
import time

def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/dashboard.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

# Give Streamlit a few seconds to start before showing the URL
time.sleep(5)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print()
print("=" * 60)
print("DASHBOARD IS RUNNING")
print("=" * 60)
print(f"Open this URL in your browser:")
print(f"  {url}")
print()
print("The URL changes every time you restart — always run this")
print("cell again after a reconnect to get the current URL.")
print("=" * 60)
